# QLoRA training for binary solution fusion

This notebook trains Qwen3-4B-Instruct-2507 on merging-dataset JSON files and evaluates base versus adapted binary fusion trees. Labels are evaluator-accepted outputs; they do not by themselves prove valid reasoning or improvement over either candidate.

In [ ]:
# Point this at the uploaded/cloned project when it is not the current directory.
from pathlib import Path
import os
# The quantized model is pinned to one device; hide extra Kaggle GPUs before importing torch.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
REPO_DIR = Path('/kaggle/working/analogical_math_rag')
if (Path.cwd() / 'src').is_dir():
    REPO_DIR = Path.cwd()
if not (REPO_DIR / 'src').is_dir():
    raise FileNotFoundError(f'Project repository not found at {REPO_DIR}')
os.chdir(REPO_DIR)
# Kaggle provides CUDA PyTorch; keep it and install only the isolated workflow dependencies.
!pip install -q -r requirements-merging-finetuning.txt


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import torch
from transformers import AutoTokenizer

from src.merging_finetuning import (
    QLoRAConfig, CompletionOnlyCollator, LocalFusionGenerator,
    build_evaluation_populations, compare_base_and_adapted_candidate_trees,
    evaluate_tree_trace, load_adapter_for_inference, load_qlora_model,
    generate_candidate_pool, prepare_merging_data, retrieve_exemplars_cpu,
    run_direct_solution, run_single_candidate_revision,
    smoke_test_training_step, summarize_evaluated_runs, tokenize_splits, train_qlora,
    upload_adapter_to_hub,
)
from src.utils import load_embedding_model, load_exemplar_corpus, load_json, save_json_atomic
from src.benchmark_data import load_target_benchmarks
from src.api_manager import AvalAIAPIManager
from config import CONFIG, setup_kaggle_mode

assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running this notebook.'
print(torch.cuda.get_device_name(0))


## Configuration
Edit these paths and run limits. Secrets are read from Kaggle Secrets or environment variables and are never printed.

In [ ]:
MERGING_JSON_PATHS = [
    '/kaggle/input/merging-dataset/merging_llama-3.2-11b_gpt-oss-20b_v1.json',
]
WORK_DIR = Path('/kaggle/working/merging-qwen3-4b-qlora')
MAX_LENGTH = 4096
SEED = 42
TRAIN = True
RESUME_CHECKPOINT = None  # e.g. '/kaggle/working/.../checkpoint-100'
EVAL_QUESTION_LIMIT = 5  # set None for the complete evaluation
RUN_PHASE_1 = True  # controlled two-candidate fusion
RUN_PHASE_2 = True  # N=4 and N=8 recursive fusion
PHASE_2_TREE_SIZES = (4, 8)
GENERATION = {'temperature': 0.7, 'top_p': 0.8, 'max_new_tokens': 1024}
HF_UPLOAD_ENABLED = True
HF_TOKEN_SECRET_NAME = 'HF_TOKEN'  # Kaggle Secret or environment variable
HF_MODEL_REPO_ID = None  # None -> <authenticated-user>/merging-qwen3-4b-qlora
HF_MODEL_REPO_PRIVATE = True
HF_UPLOAD_COMMIT_MESSAGE = 'Upload trained merging QLoRA adapter'

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f'Missing required secret {name}') from exc

qlora_config = QLoRAConfig(
    output_dir=str(WORK_DIR), max_length=MAX_LENGTH, epochs=3,
    learning_rate=1e-4, batch_size=1, gradient_accumulation_steps=8,
    lora_rank=16, lora_alpha=32, lora_dropout=0.05, seed=SEED,
)
WORK_DIR.mkdir(parents=True, exist_ok=True)


## Parse, deduplicate, split by question, and inspect token lengths

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(qlora_config.model_name, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
prepared = prepare_merging_data(
    MERGING_JSON_PATHS, WORK_DIR, tokenizer=tokenizer,
    max_length=MAX_LENGTH, seed=SEED,
)
print(json.dumps({
    'valid_records': sum(len(v) for v in prepared['splits'].values()),
    'split_sizes': {k: len(v) for k, v in prepared['splits'].items()},
    'parse_or_duplicate_failures': len(prepared['manifest']['parsing_failures']),
    'token_statistics': prepared['manifest']['token_statistics'],
}, indent=2))


## Load QLoRA model and run a real forward/backward smoke step

In [ ]:
model, tokenizer = load_qlora_model(qlora_config)
# Re-tokenize with the exact training tokenizer, then verify completion-only loss.
tokenized, token_report = tokenize_splits(prepared['splits'], tokenizer, MAX_LENGTH)
collator = CompletionOnlyCollator(tokenizer)
smoke_loss = smoke_test_training_step(model, collator, tokenized['train'][0])
print({'smoke_loss': smoke_loss, 'trainable_parameters': model.get_nb_trainable_parameters()})


## Train or resume, select the best adapter, and upload it
After successful training, the notebook uploads `best_adapter` when `HF_UPLOAD_ENABLED=True`. Add a write-enabled `HF_TOKEN` to Kaggle Secrets. The upload contains the QLoRA adapter and tokenizer files; the base model is referenced rather than duplicated.

In [ ]:
if TRAIN:
    trainer = train_qlora(
        model, tokenizer, tokenized, qlora_config,
        resume_from_checkpoint=RESUME_CHECKPOINT,
    )
    print('Best checkpoint:', trainer.state.best_model_checkpoint)
    if HF_UPLOAD_ENABLED:
        hf_token = read_secret(HF_TOKEN_SECRET_NAME)
        try:
            hub_result = upload_adapter_to_hub(
                WORK_DIR / 'best_adapter', hf_token, HF_MODEL_REPO_ID,
                private=HF_MODEL_REPO_PRIVATE,
                commit_message=HF_UPLOAD_COMMIT_MESSAGE,
            )
        finally:
            del hf_token
        print('Uploaded adapter:', hub_result['url'])


## Reload the saved adapter and configure local tree inference

In [ ]:
del model
torch.cuda.empty_cache()
model, tokenizer = load_adapter_for_inference(
    str(WORK_DIR / 'best_adapter'), qlora_config.model_name, gpu_index=0,
)
generator = LocalFusionGenerator(model, tokenizer, max_input_tokens=MAX_LENGTH)


## Load CPU retrieval resources and benchmark references
Ground truths stay in the evaluation records and are never passed to retrieval or generation.

In [ ]:
setup_kaggle_mode('/kaggle/working')
CONFIG.update({
    'TARGET_BENCHMARK': 'numina_hard', 'TARGET_BENCHMARKS': [],
    'BENCHMARK_MAX_QUESTIONS': None, 'EVAL_PARSE_BOXED_GROUND_TRUTH': True,
    'DEFAULT_EVALUATOR_TEMPERATURE': 0.0,
    'AVALAI_MODEL_NAME_EVALUATOR': 'openai/gpt-oss-20b',
})
exemplar_dataset = load_exemplar_corpus(CONFIG)
embedding_model = load_embedding_model(CONFIG)
embedded_exemplars = np.load(CONFIG['EMBEDDED_EXEMPLAR_CORPUS_QUESTIONS_PATH'])
exemplar_data = {
    'questions': list(exemplar_dataset['problem']),
    'solutions': list(exemplar_dataset['solution']),
}
benchmark_questions, benchmark_ground_truths, _ = load_target_benchmarks(
    CONFIG, exemplar_data, load_json_fn=load_json,
)
populations = build_evaluation_populations(
    prepared, benchmark_questions, benchmark_ground_truths,
)
print({name: len(rows) for name, rows in populations.items()})


## Initialize the existing AvalAI evaluator from a secret

In [ ]:
avalai_key = read_secret('AVALAI_API_KEY')
evaluator = AvalAIAPIManager(
    api_key_or_list=[avalai_key], base_url=CONFIG['AVALAI_BASE_URL'],
    model_quotas=CONFIG['AVALAI_MODEL_QUOTAS'], config=CONFIG,
)
del avalai_key


## Phase 1: controlled two-candidate fusion
For every question, the base model creates a fixed pair from each source: two independent zero-shot samples, or two independent one-shot solutions made from R1 and R2 separately. The notebook compares direct solving, revision of each candidate alone, and fusion of the identical pair with both the base and adapted model arms.

In [ ]:
if any(size < 2 or size & (size - 1) for size in PHASE_2_TREE_SIZES):
    raise ValueError('PHASE_2_TREE_SIZES must contain powers of two >= 2')
max_candidate_count = max((2,) + tuple(PHASE_2_TREE_SIZES if RUN_PHASE_2 else ()))
evaluation_cache = {}

def attach_evaluation(tree, ground_truth):
    return evaluate_tree_trace(
        tree, ground_truth, evaluator, CONFIG, evaluation_cache=evaluation_cache,
    )

def make_run(phase, population, record, source, count, mode, arm, tree):
    return {
        'phase': phase, 'population': population,
        'benchmark_index': record['benchmark_index'],
        'candidate_source': source, 'candidate_count': count,
        'mode': mode, 'arm': arm, 'tree': tree,
        'evaluation': attach_evaluation(tree, record['ground_truth']),
    }

def evaluate_population(name, records, limit=None):
    phase_1_runs, phase_2_runs = [], []
    selected = records if limit is None else records[:limit]
    for question_index, record in enumerate(selected):
        question = record['question']
        question_seed = SEED + question_index * 10000
        retrieved = retrieve_exemplars_cpu(
            question, exemplar_data['questions'], exemplar_data['solutions'],
            embedded_exemplars, embedding_model, top_k=max_candidate_count,
        )
        pools = {
            'zero_shot': generate_candidate_pool(
                question, generator, 'zero_shot', count=max_candidate_count,
                seed=question_seed, generation=GENERATION,
            ),
            'one_shot': generate_candidate_pool(
                question, generator, 'one_shot', count=max_candidate_count,
                retrieved_examples=retrieved, seed=question_seed + 1000,
                generation=GENERATION,
            ),
        }

        if RUN_PHASE_1:
            for arm, use_adapter in [('base', False), ('adapted', True)]:
                direct = run_direct_solution(
                    question, generator, use_adapter=use_adapter,
                    seed=question_seed + 2000, generation=GENERATION,
                )
                phase_1_runs.append(make_run(
                    'phase_1', name, record, 'none', 0,
                    'direct_solution', arm, direct,
                ))
            for source, pool in pools.items():
                for candidate_index, candidate in enumerate(pool['candidates'][:2]):
                    for arm, use_adapter in [('base', False), ('adapted', True)]:
                        revision = run_single_candidate_revision(
                            question, candidate, generator, use_adapter=use_adapter,
                            seed=question_seed + 2500 + candidate_index,
                            generation=GENERATION,
                        )
                        phase_1_runs.append(make_run(
                            'phase_1', name, record, source, 1,
                            f'single_candidate_{candidate_index + 1}', arm, revision,
                        ))
                compared = compare_base_and_adapted_candidate_trees(
                    question, pool['candidates'][:2], generator,
                    seed=question_seed + 3000, generation=GENERATION,
                )
                for arm, tree in compared.items():
                    phase_1_runs.append(make_run(
                        'phase_1', name, record, source, 2,
                        'pair_fusion', arm, tree,
                    ))

        if RUN_PHASE_2:
            for source, pool in pools.items():
                for count in PHASE_2_TREE_SIZES:
                    compared = compare_base_and_adapted_candidate_trees(
                        question, pool['candidates'][:count], generator,
                        seed=question_seed + 4000,
                        generation=GENERATION,
                    )
                    for arm, tree in compared.items():
                        phase_2_runs.append(make_run(
                            'phase_2', name, record, source, count,
                            'tree_fusion', arm, tree,
                        ))

        for phase, runs in [('phase_1', phase_1_runs), ('phase_2', phase_2_runs)]:
            if runs and not save_json_atomic(
                runs, str(WORK_DIR / f'{phase}_{name}_results.json')
            ):
                raise OSError(f'Failed to checkpoint {phase} results')
    return phase_1_runs, phase_2_runs

phase_1_runs, phase_2_runs = [], []
for population_name, records in populations.items():
    p1, p2 = evaluate_population(population_name, records, EVAL_QUESTION_LIMIT)
    phase_1_runs.extend(p1)
    phase_2_runs.extend(p2)
all_runs = phase_1_runs + phase_2_runs


## Phase 2: recursive fusion at N=4 and N=8
Phase 2 uses prefixes of the same saved candidate pools, so N=4 is nested inside N=8. Summaries report root accuracy, input-candidate accuracy, candidate-correctness strata, fusion accuracy at each layer, and paired adapted-minus-base differences with question-level bootstrap intervals.

In [ ]:
def diagnostic_summary(runs):
    result = summarize_evaluated_runs(runs)
    candidate_values, layer_values = [], {}
    strata = {name: [] for name in (
        'both_correct', 'one_correct', 'neither_correct',
        'candidate_correct', 'candidate_incorrect',
        'all_correct', 'some_correct', 'none_correct', 'unknown',
    )}
    for run in runs:
        correctness = run['evaluation'].get('node_correctness', {})
        candidate_nodes = [
            node for node in run['tree'].get('trace', [])
            if node.get('kind', '').endswith('_candidate')
        ]
        values = [correctness.get(node['node_id']) for node in candidate_nodes]
        candidate_values.extend(value for value in values if value is not None)
        if values:
            if any(value is None for value in values):
                stratum = 'unknown'
            elif len(values) == 1:
                stratum = 'candidate_correct' if values[0] else 'candidate_incorrect'
            elif len(values) == 2:
                stratum = 'both_correct' if all(values) else (
                    'one_correct' if any(values) else 'neither_correct'
                )
            else:
                stratum = 'all_correct' if all(values) else (
                    'some_correct' if any(values) else 'none_correct'
                )
            root = run['evaluation'].get('root_correct')
            if root is not None:
                strata[stratum].append(root)
        for node in run['tree'].get('trace', []):
            if node.get('kind') == 'fusion':
                layer = int(node['node_id'].split('-')[1])
                value = correctness.get(node['node_id'])
                if value is not None:
                    layer_values.setdefault(layer, []).append(value)
    result['candidate_accuracy'] = (
        sum(candidate_values) / len(candidate_values) if candidate_values else None
    )
    result['root_accuracy_by_candidate_stratum'] = {
        name: {'runs': len(values), 'accuracy': sum(values) / len(values) if values else None}
        for name, values in strata.items()
    }
    result['fusion_accuracy_by_layer'] = {
        str(layer): {'nodes': len(values), 'accuracy': sum(values) / len(values)}
        for layer, values in sorted(layer_values.items())
    }
    return result

def paired_arm_effect(runs, bootstrap_samples=10000):
    by_question = {}
    for run in runs:
        value = run.get('evaluation', {}).get('root_correct')
        if value is not None:
            by_question.setdefault(run['benchmark_index'], {})[run['arm']] = int(value)
    pairs = [arms for arms in by_question.values() if {'base', 'adapted'} <= arms.keys()]
    if not pairs:
        return {'paired_questions': 0, 'accuracy_delta': None, 'bootstrap_95_ci': None}
    differences = np.asarray([pair['adapted'] - pair['base'] for pair in pairs], dtype=float)
    rng = np.random.default_rng(SEED)
    boot = differences[rng.integers(0, len(differences), size=(bootstrap_samples, len(differences)))].mean(axis=1)
    return {
        'paired_questions': len(pairs),
        'base_accuracy': float(np.mean([pair['base'] for pair in pairs])),
        'adapted_accuracy': float(np.mean([pair['adapted'] for pair in pairs])),
        'accuracy_delta': float(differences.mean()),
        'bootstrap_95_ci': [float(value) for value in np.quantile(boot, [0.025, 0.975])],
    }

summary, paired_effects = {}, {}
group_fields = ('phase', 'population', 'candidate_source', 'candidate_count', 'mode')
group_keys = sorted({tuple(run[field] for field in group_fields) for run in all_runs})
for group_key in group_keys:
    grouped = [
        run for run in all_runs
        if tuple(run[field] for field in group_fields) == group_key
    ]
    group_name = '/'.join(map(str, group_key))
    for arm in ('base', 'adapted'):
        arm_runs = [run for run in grouped if run['arm'] == arm]
        if arm_runs:
            summary[f'{group_name}/{arm}'] = diagnostic_summary(arm_runs)
    paired_effects[group_name] = paired_arm_effect(grouped)

outputs = {
    'phase_1_runs': len(phase_1_runs), 'phase_2_runs': len(phase_2_runs),
    'summaries': summary, 'paired_adapted_minus_base': paired_effects,
}
if not save_json_atomic(outputs, str(WORK_DIR / 'two_phase_evaluation_summary.json')):
    raise OSError('Failed to save two-phase evaluation summary')
print(json.dumps(outputs, indent=2))
